In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import csv
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import random

The textconverter.csv file contains all the links on https://scrapsfromtheloft.com/stand-up-comedy-scripts/ containing 'comedy' and 'transcript'. The process was:
1. Extract links from site using https://simplescraper.io/extracturls and filter for the word "comedy"
2. Put the resulting txt file into https://textconverter.com/filter-text and filter for the word "transcript"
3. Put the resulting txt file into https://convertio.co/txt-csv/ to convert it to a csv

In [ ]:
links_csv = "/textconverter.csv"
urls = pd.read_csv(links_csv, header=None)[0].tolist()
session = None
headers = None

In [4]:
df_json = "/content/4300_articles.json"
df = pd.read_json(df_json, lines=True)
df

,url,title,content
0,https://scrapsfromtheloft.com/comedy/gary-gulm...,Gary Gulman: The Great Depresh (2019) | Transc...,Gary Gulman\n‘s 2019 HBO stand-up comedy speci...
1,https://scrapsfromtheloft.com/comedy/ricky-ger...,Ricky Gervais: Armageddon (2023) | Transcript,Ricky Gervais: Armageddon\nis packed with\nGer...
2,https://scrapsfromtheloft.com/comedy/gary-gulm...,Gary Gulman: It’s About Time (2016) | Transcript,Imagine stepping into the mind of a comedic wh...
3,https://scrapsfromtheloft.com/comedy/orny-adam...,None,None
4,https://scrapsfromtheloft.com/comedy/dave-chap...,Dave Chappelle: The Dreamer (2023) | Transcript,"“The Dreamer,” which was shot in\nChappelle\n’..."
...,...,...,...
588,https://scrapsfromtheloft.com/comedy/sebastian...,None,None
589,https://scrapsfromtheloft.com/comedy/rory-scov...,None,None
590,https://scrapsfromtheloft.com/comedy/robby-hof...,None,None
591,https://scrapsfromtheloft.com/comedy/robby-hof...,None,None


In [5]:
df_missing = df[df['title'].isna()]
urls_missing = df_missing['url'].to_list()
urls_missing

['https://scrapsfromtheloft.com/comedy/orny-adams-more-than-loud-transcript/',
 'https://scrapsfromtheloft.com/comedy/lisa-lampanelli-back-to-the-drawing-board-2015-full-transcript/',
 'https://scrapsfromtheloft.com/comedy/jeff-foxworthy-totally-committed-transcript/',
 'https://scrapsfromtheloft.com/comedy/election-results-2020-last-week-tonight-with-john-oliver-transcript/',
 'https://scrapsfromtheloft.com/comedy/dave-chappelle-worth-2004-full-transcript/',
 'https://scrapsfromtheloft.com/comedy/dave-chappelle-equanimity-2017-full-transcript/',
 'https://scrapsfromtheloft.com/comedy/roy-wood-jr-father-figure-transcript/',
 'https://scrapsfromtheloft.com/comedy/amanda-seales-i-be-knowin-transcript/',
 'https://scrapsfromtheloft.com/comedy/arsenio-hall-smart-and-classy-transcript/',
 'https://scrapsfromtheloft.com/comedy/aziz-ansari-right-now-transcript/',
 'https://scrapsfromtheloft.com/comedy/the-standups-gina-yashere-2018-transcript/',
 'https://scrapsfromtheloft.com/comedy/dave-cha

In [ ]:
def setup_session():
  session = requests.Session()

  headers = {
      "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
  }

  session.headers.update(headers)

In [6]:
def scrape_page(link):
    try:
        time.sleep(random.uniform(2, 5))

        response = session.get(link, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        title_tag = soup.find("h1", class_="elementor-heading-title")
        title = title_tag.get_text(strip=True) if title_tag else None

        """
        <div class="elementor-element elementor-element-74af9a5b elementor-widget elementor-widget-theme-post-content" data-id="74af9a5b" data-element_type="widget" data-widget_type="theme-post-content.default" style="height: auto !important;">
        """
        content_div = soup.find("div", {"data-id": "74af9a5b"})
        content = content_div.get_text(separator="\n", strip=True) if content_div else None

        return {
            "url": link,
            "title": title,
            "content": content
        }

    except Exception as e:
        print(f"Failed: {link} | {e}")
        return {"url": link, "title": None, "content": None}

In [ ]:
records = []

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

session.headers.update(headers)

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = [executor.submit(scrape_page, url) for url in urls]

    for future in as_completed(futures):
        records.append(future.result())

df = pd.DataFrame(records)

Failed: https://scrapsfromtheloft.com/comedy/orny-adams-more-than-loud-transcript/ | ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Failed: https://scrapsfromtheloft.com/comedy/lisa-lampanelli-back-to-the-drawing-board-2015-full-transcript/ | HTTPSConnectionPool(host='scrapsfromtheloft.com', port=443): Read timed out. (read timeout=10)
Failed: https://scrapsfromtheloft.com/comedy/jeff-foxworthy-totally-committed-transcript/ | ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Failed: https://scrapsfromtheloft.com/comedy/election-results-2020-last-week-tonight-with-john-oliver-transcript/ | 503 Server Error: Service Temporarily Unavailable for url: https://scrapsfromtheloft.com/comedy/election-results-2020-last-week-tonight-with-john-oliver-transcript/
Failed: https://scrapsfromtheloft.com/comedy/dave-chappelle-worth-2004-full-transcript/ | 503 Server Error: Service Unavailable for url: https://scrap

In [8]:
records = []

session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

session.headers.update(headers)

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = [executor.submit(scrape_page, url) for url in urls_missing]

    for future in as_completed(futures):
        records.append(future.result())

df_filled = pd.DataFrame(records)

Failed: https://scrapsfromtheloft.com/comedy/sacha-baron-cohens-who-is-america-episode-6-transcript/ | ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


In [9]:
df_filled

,url,title,content
0,https://scrapsfromtheloft.com/comedy/jeff-foxw...,Jeff Foxworthy: Totally Committed (1998) – Ful...,"Ladies and gentlemen, please welcome Jeff Foxw..."
1,https://scrapsfromtheloft.com/comedy/lisa-lamp...,LISA LAMPANELLI: BACK TO THE DRAWING BOARD (20...,"Ladies and gentlemen, please welcome Lisa Lamp..."
2,https://scrapsfromtheloft.com/comedy/orny-adam...,Orny Adams: More Than Loud (2017) | Transcript,Title:\nOrny Adams: More Than Loud\nType:\nSta...
3,https://scrapsfromtheloft.com/comedy/election-...,Election Results 2020: Last Week Tonight with ...,Last Week Tonight with John Oliver\nSeason 7 E...
4,https://scrapsfromtheloft.com/comedy/dave-chap...,Dave Chappelle: Equanimity (2017) – Transcript,"“Equanimity” was shot in Washington, D.C., and..."
...,...,...,...
123,https://scrapsfromtheloft.com/comedy/deon-cole...,"Deon Cole: Ok, Mister (2024) | Transcript",[“Post That” by Deon Cole and Terry Hunter pla...
124,https://scrapsfromtheloft.com/comedy/sebastian...,Sebastian Maniscalco: It Ain’t Right (2025) | ...,Sebastian Maniscalco: It Ain’t Right (2025)\nD...
125,https://scrapsfromtheloft.com/comedy/rory-scov...,Rory Scovel on Religion,Rory Scovel\non\nreligion\nOne time my daughte...
126,https://scrapsfromtheloft.com/comedy/robby-hof...,Robby Hoffman: I’m Nervous (2019) | Transcript,Robby Hoffman: I’m Nervous\nis the debut one-h...


In [ ]:
# records = []

# for url in urls:
#     record = scrape_page(url)
#     records.append(record)
#     time.sleep(random.uniform(0.5, 2))


# df = pd.DataFrame(records)

Failed: https://scrapsfromtheloft.com/comedy/bill-maher-glad-david-koch-is-dead-transcript/ | 503 Server Error: Service Unavailable for url: https://scrapsfromtheloft.com/comedy/bill-maher-glad-david-koch-is-dead-transcript/
Failed: https://scrapsfromtheloft.com/comedy/ryan-hamilton-stand-up-the-tonight-show-starring-jimmy-fallon-transcript/ | 503 Server Error: Service Unavailable for url: https://scrapsfromtheloft.com/comedy/ryan-hamilton-stand-up-the-tonight-show-starring-jimmy-fallon-transcript/
Failed: https://scrapsfromtheloft.com/comedy/sebastian-maniscalco-tonight-show-starring-jimmy-fallon-transcript/ | 503 Server Error: Service Unavailable for url: https://scrapsfromtheloft.com/comedy/sebastian-maniscalco-tonight-show-starring-jimmy-fallon-transcript/
Failed: https://scrapsfromtheloft.com/comedy/seth-meyers-2011-white-house-correspondents-dinner-transcript/ | 503 Server Error: Service Unavailable for url: https://scrapsfromtheloft.com/comedy/seth-meyers-2011-white-house-corres

KeyboardInterrupt: 

In [10]:
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/120.0 Safari/537.36"
}

session.headers.update(headers)

last_link = 'https://scrapsfromtheloft.com/comedy/sacha-baron-cohens-who-is-america-episode-6-transcript/'

last_data = scrape_page(last_link)
last_data

{'url': 'https://scrapsfromtheloft.com/comedy/sacha-baron-cohens-who-is-america-episode-6-transcript/',
 'title': 'Sacha Baron Cohen’s Who is America? – Episode 6 – Transcript',
 'content': 'Scenes:\nBilly Wayne Ruddick Jr., PhD interviews former presidential candidate Jill Stein and discusses climate change and global warming.\nHe then interviews former Governor of Vermont Howard Dean and discusses his theory that Hillary Clinton is secretly a man.\nErran Morad teaches reality personality Gretchen Rossi and her husband Slade Smiley how to protect themselves from home invasion.\nDr. Nira Cain-N’Degeocello travels to the Las Vegas Enlightenment Center and meets with spiritual healer Ataana Badilli. Cain-N’Degeocello discusses his plan to “give birth” to a baby doll from his rectum as a means empathizing with his wife whom he recently impregnated. While going through the procedure, Cain-N’Degeocello’s maid Maria acts as his doula and attempts to remove the head of the doll from him as it

In [28]:
df_filled.loc[df_filled['url'] == last_data['url'], ['title', 'content']] = [last_data['title'], last_data['content']]

In [29]:
df_filled[df_filled['url'] == last_data['url']]

,url,title,content
64,https://scrapsfromtheloft.com/comedy/sacha-bar...,Sacha Baron Cohen’s Who is America? – Episode ...,"Scenes:\nBilly Wayne Ruddick Jr., PhD intervie..."


In [37]:
df

,title,content
url,,
https://scrapsfromtheloft.com/comedy/jeff-foxworthy-totally-committed-transcript/,Jeff Foxworthy: Totally Committed (1998) – Ful...,"Ladies and gentlemen, please welcome Jeff Foxw..."
https://scrapsfromtheloft.com/comedy/lisa-lampanelli-back-to-the-drawing-board-2015-full-transcript/,LISA LAMPANELLI: BACK TO THE DRAWING BOARD (20...,"Ladies and gentlemen, please welcome Lisa Lamp..."
https://scrapsfromtheloft.com/comedy/orny-adams-more-than-loud-transcript/,Orny Adams: More Than Loud (2017) | Transcript,Title:\nOrny Adams: More Than Loud\nType:\nSta...
https://scrapsfromtheloft.com/comedy/election-results-2020-last-week-tonight-with-john-oliver-transcript/,Election Results 2020: Last Week Tonight with ...,Last Week Tonight with John Oliver\nSeason 7 E...
https://scrapsfromtheloft.com/comedy/dave-chappelle-equanimity-2017-full-transcript/,Dave Chappelle: Equanimity (2017) – Transcript,"“Equanimity” was shot in Washington, D.C., and..."
...,...,...
https://scrapsfromtheloft.com/comedy/sebastian-maniscalco-it-aint-right-transcript/,None,None
https://scrapsfromtheloft.com/comedy/rory-scovel-on-religion-transcript/,None,None
https://scrapsfromtheloft.com/comedy/robby-hoffman-im-nervous-transcript/,None,None


In [38]:
df_filled = df_filled.set_index("url")

df.update(df_filled[['title', 'content']])

df = df.reset_index()
df

,url,title,content
0,https://scrapsfromtheloft.com/comedy/jeff-foxw...,Jeff Foxworthy: Totally Committed (1998) – Ful...,"Ladies and gentlemen, please welcome Jeff Foxw..."
1,https://scrapsfromtheloft.com/comedy/lisa-lamp...,LISA LAMPANELLI: BACK TO THE DRAWING BOARD (20...,"Ladies and gentlemen, please welcome Lisa Lamp..."
2,https://scrapsfromtheloft.com/comedy/orny-adam...,Orny Adams: More Than Loud (2017) | Transcript,Title:\nOrny Adams: More Than Loud\nType:\nSta...
3,https://scrapsfromtheloft.com/comedy/election-...,Election Results 2020: Last Week Tonight with ...,Last Week Tonight with John Oliver\nSeason 7 E...
4,https://scrapsfromtheloft.com/comedy/dave-chap...,Dave Chappelle: Equanimity (2017) – Transcript,"“Equanimity” was shot in Washington, D.C., and..."
...,...,...,...
588,https://scrapsfromtheloft.com/comedy/sebastian...,Sebastian Maniscalco: It Ain’t Right (2025) | ...,Sebastian Maniscalco: It Ain’t Right (2025)\nD...
589,https://scrapsfromtheloft.com/comedy/rory-scov...,Rory Scovel on Religion,Rory Scovel\non\nreligion\nOne time my daughte...
590,https://scrapsfromtheloft.com/comedy/robby-hof...,Robby Hoffman: I’m Nervous (2019) | Transcript,Robby Hoffman: I’m Nervous\nis the debut one-h...
591,https://scrapsfromtheloft.com/comedy/robby-hof...,Robby Hoffman: Wake Up (2025) | Transcript,Robby Hoffman: Wake Up\nis the second hour-lon...


In [36]:
df.columns

Index(['title', 'content'], dtype='object')

In [39]:
df[df['title'].isna()]

,url,title,content


In [40]:
df.to_json("4300_transcripts.json", orient="records", lines=True)

In [43]:
df.loc[444, 'content']

"In this comedy special taped at DAR Constitution Hall, his first solo special on the network in seven years, Williams covers such topics as global warming, sex and politics, the state of health care in the country (suggesting a cash for clunkers program for elderly relatives, among other things), drugs – recreational and otherwise – and more personal topics, including his recent heart surgery.\nOriginal Air Date on December 6, 2009\n* * *\n[audience cheering, applauding]\n[rock music playing]\nAnnouncer: Ladies and gentlemen, Please welcome Robin Williams!\n[cheering]\nNo! Please. Sit down! Thank you! Thank you! What’s up, D.C.? Yes, indeed! Yes, indeed! [cheering, hooting] Wow, thank you. Mmm. Thank you. Please, I’ve had heart surgery. Thank you. It’s nice to be in Washington, where the buck stops here. Way to go. And then it’s handed out to A.I.G. and many other people. Now… I have the new Timothy Geithner $20-bill. It’s just been printed, kind of neat. Instead of “in god we trust,”